# Tennis Point-by-Point Preprocessing

## 1. Setup and locate the extracted point-by-point data

In [1]:
# Import the libraries needed for point-by-point preprocessing

from pathlib import Path
import polars as pl

In [2]:
# Locate the extracted tennis dataset from the preprocessing folder

MINI_PROJECT_ROOT = Path.cwd().parents[2]

DATA_ROOT = (
    MINI_PROJECT_ROOT
    / "Tennis Schema"
    / "tennis_data"
)

EXTRACT_ROOT = DATA_ROOT / "extracted"

print(EXTRACT_ROOT)
print("Exists:", EXTRACT_ROOT.exists())

d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data\extracted
Exists: True


In [3]:
# Collect every extracted point-by-point Parquet file
# Each daily folder represents a separate snapshot of the available data

point_files = sorted(
    EXTRACT_ROOT.glob(
        "*/raw_point_by_point_parquet/*.parquet"
    )
)

print("Point-by-point file appearances:", len(point_files))

Point-by-point file appearances: 22272


In [4]:
# Preview a few paths to confirm that the correct folder was selected

point_files[:5]

[WindowsPath('d:/Learning/Daneshkar/Statistics/Stat with Python/Mini Project/Tennis Schema/tennis_data/extracted/20240201/raw_point_by_point_parquet/pbp_11998445.parquet'),
 WindowsPath('d:/Learning/Daneshkar/Statistics/Stat with Python/Mini Project/Tennis Schema/tennis_data/extracted/20240201/raw_point_by_point_parquet/pbp_11998446.parquet'),
 WindowsPath('d:/Learning/Daneshkar/Statistics/Stat with Python/Mini Project/Tennis Schema/tennis_data/extracted/20240201/raw_point_by_point_parquet/pbp_11998447.parquet'),
 WindowsPath('d:/Learning/Daneshkar/Statistics/Stat with Python/Mini Project/Tennis Schema/tennis_data/extracted/20240201/raw_point_by_point_parquet/pbp_11998448.parquet'),
 WindowsPath('d:/Learning/Daneshkar/Statistics/Stat with Python/Mini Project/Tennis Schema/tennis_data/extracted/20240201/raw_point_by_point_parquet/pbp_11998449.parquet')]


## 2. Inspect the point-by-point data structure

In [5]:
# Read one point-by-point file as a sample
# Inspect the raw data before making any cleaning or tennis-rule decisions

sample_file = point_files[0]

point_sample = pl.read_parquet(sample_file)

print("Sample file:", sample_file.name)
print("Shape:", point_sample.shape)

Sample file: pbp_11998445.parquet
Shape: (159, 13)


In [6]:
# Inspect the column names and data types in the sample file

point_sample.schema

Schema([('match_id', Int64),
        ('set_id', Int64),
        ('game_id', Int64),
        ('point_id', Int64),
        ('home_point', String),
        ('away_point', String),
        ('point_description', Int64),
        ('home_point_type', Int64),
        ('away_point_type', Int64),
        ('home_score', Int64),
        ('away_score', Int64),
        ('serving', Int64),
        ('scoring', Int64)])

In [7]:
# Preview the first rows to understand how sets, games, points, and tennis scores are represented in the raw data

point_sample.head(10)

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
11998445,3,13,0,"""1""","""0""",0,6,5,6,7,1,2
11998445,3,13,1,"""1""","""1""",0,5,6,6,7,1,2
11998445,3,13,2,"""1""","""2""",0,5,6,6,7,1,2
11998445,3,13,3,"""1""","""3""",0,5,1,6,7,1,2
11998445,3,13,4,"""1""","""4""",0,5,1,6,7,1,2
11998445,3,13,5,"""2""","""4""",0,1,5,6,7,1,2
11998445,3,13,6,"""3""","""4""",0,1,5,6,7,1,2
11998445,3,13,7,"""3""","""5""",0,5,1,6,7,1,2
11998445,3,13,8,"""4""","""5""",0,6,5,6,7,1,2


In [8]:
# Inspect the end of the sample match to see how the final set and game conclude

point_sample.tail(10)

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
11998445,1,4,4,"""40""","""30""",0,1,5,2,2,1,1
11998445,1,3,0,"""0""","""15""",0,5,1,1,2,2,2
11998445,1,3,1,"""0""","""30""",0,5,1,1,2,2,2
11998445,1,3,2,"""0""","""40""",0,5,1,1,2,2,2
11998445,1,2,0,"""15""","""0""",0,1,5,1,1,1,1
11998445,1,2,1,"""30""","""0""",0,1,5,1,1,1,1
11998445,1,2,2,"""40""","""0""",1,1,5,1,1,1,1
11998445,1,1,0,"""0""","""15""",0,5,1,0,1,2,2
11998445,1,1,1,"""0""","""30""",0,5,1,0,1,2,2


In [9]:
# Summarize the games contained in the sample match
# This helps us see how game numbering and set progression are represented

point_sample.group_by(
    ["set_id", "game_id"]
).agg(
    pl.len().alias("number_of_rows"),
    pl.col("home_score").first().alias("home_score"),
    pl.col("away_score").first().alias("away_score")
).sort(
    ["set_id", "game_id"]
)

set_id,game_id,number_of_rows,home_score,away_score
i64,i64,u32,i64,i64
1,1,3,0,1
1,2,3,1,1
1,3,3,1,2
1,4,5,2,2
1,5,5,3,2
…,…,…,…,…
3,9,4,5,4
3,10,3,5,5
3,11,5,5,6


In [10]:
# Sort the sample into chronological tennis order
# The raw file stores later sets/games before earlier ones

point_sample_sorted = point_sample.sort(
    ["set_id", "game_id", "point_id"]
)

point_sample_sorted.head(20)

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
11998445,1,1,0,"""0""","""15""",0,5,1,0,1,2,2
11998445,1,1,1,"""0""","""30""",0,5,1,0,1,2,2
11998445,1,1,2,"""0""","""40""",0,5,1,0,1,2,2
11998445,1,2,0,"""15""","""0""",0,1,5,1,1,1,1
11998445,1,2,1,"""30""","""0""",0,1,5,1,1,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…
11998445,1,5,1,"""15""","""15""",0,5,1,3,2,2,1
11998445,1,5,2,"""30""","""15""",0,1,5,3,2,2,1
11998445,1,5,3,"""40""","""15""",0,2,5,3,2,2,1


## 3. Investigate schema consistency and missing values

### 3.1 Check schema consistency
Before combining the point-by-point files, we check whether all files have the same structure and whether important fields contain missing values. This prevents schema differences or incomplete records from causing problems during preprocessing.

In [11]:
# Check whether all point-by-point files have the same column names and data types
# We do this before combining the files so schema differences can be handled safely

schema_groups = {}

for file in point_files:
    schema = tuple(pl.read_parquet_schema(file).items())

    if schema not in schema_groups:
        schema_groups[schema] = []

    schema_groups[schema].append(file)

print("Number of different schemas:", len(schema_groups))

Number of different schemas: 1


### 3.2 Check missing values

In [12]:
# Count missing values in each column across one representative sample
# This gives us an initial view of which fields may need special handling

point_sample.null_count()

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0


### 3.3 Check missing values across all point-by-point files

The sample file contains no missing values, but the same check must be performed across the complete dataset before deciding whether missing-value preprocessing is necessary.

In [13]:
# Scan all point-by-point files without first loading the entire dataset into memory
# This allows us to count missing values across every daily snapshot efficiently

point_scan = pl.scan_parquet(
    [str(file) for file in point_files]
)

full_null_counts = point_scan.select(
    [
        pl.col(column).null_count().alias(column)
        for column in point_sample.columns
    ]
).collect()

full_null_counts

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0


## Stage 3 findings

All 22,272 point-by-point file appearances use the same 13-column schema.  
No missing values were found in any column across the complete dataset.

Therefore, no schema harmonization or missing-value treatment is required at this stage.

## 4. Investigate match, set, game, and point ordering

### 4.1 Check the raw row order

In [14]:
# Check whether the sample file is already stored in chronological tennis order
# The expected order is set -> game -> point

raw_order = point_sample.select(
    ["set_id", "game_id", "point_id"]
)

sorted_order = point_sample.sort(
    ["set_id", "game_id", "point_id"]
).select(
    ["set_id", "game_id", "point_id"]
)

print("Already chronological:", raw_order.equals(sorted_order))

Already chronological: False


The raw point-by-point rows are not stored in chronological order.

Therefore, all rule-based validation must first sort the data by:

`set_id -> game_id -> point_id`

Otherwise, valid tennis sequences could be incorrectly flagged as invalid.

### 4.2 Check point numbering within games

In [15]:
# Check whether point_id starts at 0 and increases without gaps inside each game
# This helps verify that the point sequence is structurally complete

point_id_check = (
    point_sample
    .sort(["set_id", "game_id", "point_id"])
    .group_by(["set_id", "game_id"])
    .agg(
        pl.col("point_id").min().alias("min_point_id"),
        pl.col("point_id").max().alias("max_point_id"),
        pl.col("point_id").n_unique().alias("unique_point_ids"),
        pl.len().alias("row_count")
    )
    .with_columns(
        (
            (pl.col("min_point_id") == 0)
            &
            (pl.col("unique_point_ids") == pl.col("row_count"))
            &
            (pl.col("max_point_id") == pl.col("row_count") - 1)
        ).alias("valid_point_sequence")
    )
)

point_id_check

set_id,game_id,min_point_id,max_point_id,unique_point_ids,row_count,valid_point_sequence
i64,i64,i64,i64,u32,u32,bool
1,1,0,2,3,3,true
2,1,0,4,5,5,true
2,3,0,3,4,4,true
2,5,0,4,5,5,true
1,5,0,4,5,5,true
…,…,…,…,…,…,…
3,5,0,3,4,4,true
2,8,0,6,7,7,true
1,7,0,4,5,5,true


In [16]:
point_id_check.filter(
    ~pl.col("valid_point_sequence")
)

set_id,game_id,min_point_id,max_point_id,unique_point_ids,row_count,valid_point_sequence
i64,i64,i64,i64,u32,u32,bool


All games in the sample match have continuous and unique point numbering starting from 0.

No gaps or duplicate `point_id` values were found within the sample games.

### 4.3 Check game numbering within sets

In [17]:
# Check whether game_id starts at 1 and increases without gaps inside each set

game_id_check = (
    point_sample
    .group_by("set_id")
    .agg(
        pl.col("game_id").min().alias("min_game_id"),
        pl.col("game_id").max().alias("max_game_id"),
        pl.col("game_id").n_unique().alias("unique_game_ids")
    )
    .with_columns(
        (
            (pl.col("min_game_id") == 1)
            &
            (pl.col("unique_game_ids") == pl.col("max_game_id"))
        ).alias("valid_game_sequence")
    )
    .sort("set_id")
)

game_id_check

set_id,min_game_id,max_game_id,unique_game_ids,valid_game_sequence
i64,i64,i64,u32,bool
1,1,12,12,true
2,1,8,8,true
3,1,13,13,true


All sets in the sample match contain continuous game numbering starting from 1.

No missing `game_id` values were found within the sample sets.

### 4.4 Check raw row order across all files

In [18]:
# Check whether every raw point-by-point file is already stored in chronological order
# Chronological order within a match is defined as set -> game -> point

unordered_files = []

for file in point_files:
    df = pl.read_parquet(file)

    raw_order = df.select(
        ["set_id", "game_id", "point_id"]
    )

    sorted_order = df.sort(
        ["set_id", "game_id", "point_id"]
    ).select(
        ["set_id", "game_id", "point_id"]
    )

    if not raw_order.equals(sorted_order):
        unordered_files.append(file)

print("Total files:", len(point_files))
print("Files not stored chronologically:", len(unordered_files))

Total files: 22272
Files not stored chronologically: 22255


The raw row order is not chronological in almost all point-by-point files.

Out of 22,272 file appearances, 22,255 are not stored in `set_id -> game_id -> point_id` order.

Therefore, every file must be sorted before validating point progression or tennis scoring rules.

### 4.5 Validate structural ordering across the complete dataset

In [19]:
# Check point_id continuity inside every game across all point-by-point files
# Each game should contain unique point IDs starting at 0 with no gaps

invalid_point_sequences = []

for file in point_files:
    df = pl.read_parquet(file)

    check = (
        df.group_by(["match_id", "set_id", "game_id"])
        .agg(
            pl.col("point_id").min().alias("min_point_id"),
            pl.col("point_id").max().alias("max_point_id"),
            pl.col("point_id").n_unique().alias("unique_point_ids"),
            pl.len().alias("row_count")
        )
        .filter(
            (pl.col("min_point_id") != 0)
            |
            (pl.col("unique_point_ids") != pl.col("row_count"))
            |
            (pl.col("max_point_id") != pl.col("row_count") - 1)
        )
    )

    if check.height > 0:
        invalid_point_sequences.append((file, check))

print("Files with invalid point_id sequences:", len(invalid_point_sequences))

Files with invalid point_id sequences: 0


No invalid `point_id` sequences were found across the complete point-by-point dataset.

Every game starts at `point_id = 0`, contains unique point identifiers, and has no gaps in the point sequence.

### 4.6 Validate game numbering across the complete dataset

In [20]:
# Check game_id continuity inside every set across all point-by-point files
# Each set should contain consecutive game IDs starting from 1

invalid_game_sequences = []

for file in point_files:
    df = pl.read_parquet(file)

    check = (
        df.group_by(["match_id", "set_id"])
        .agg(
            pl.col("game_id").min().alias("min_game_id"),
            pl.col("game_id").max().alias("max_game_id"),
            pl.col("game_id").n_unique().alias("unique_game_ids")
        )
        .filter(
            (pl.col("min_game_id") != 1)
            |
            (pl.col("unique_game_ids") != pl.col("max_game_id"))
        )
    )

    if check.height > 0:
        invalid_game_sequences.append((file, check))

print("Files with invalid game_id sequences:", len(invalid_game_sequences))

Files with invalid game_id sequences: 15


In [21]:
# Summarize the sets that failed the game_id continuity check
# We inspect them before deciding whether they represent bad data or partial match snapshots

invalid_game_details = []

for file, check in invalid_game_sequences:
    details = check.with_columns(
        pl.lit(file.name).alias("file_name"),
        pl.lit(file.parent.parent.name).alias("snapshot_date")
    )

    invalid_game_details.append(details)

invalid_game_df = (
    pl.concat(invalid_game_details)
    .select(
        [
            "snapshot_date",
            "file_name",
            "match_id",
            "set_id",
            "min_game_id",
            "max_game_id",
            "unique_game_ids"
        ]
    )
    .sort(["snapshot_date", "match_id", "set_id"])
)

invalid_game_df

snapshot_date,file_name,match_id,set_id,min_game_id,max_game_id,unique_game_ids
str,str,i64,i64,i64,i64,u32
"""20240202""","""pbp_12027665.parquet""",12027665,1,4,9,6
"""20240202""","""pbp_12027669.parquet""",12027669,1,4,12,9
"""20240202""","""pbp_12027679.parquet""",12027679,1,1,7,6
"""20240203""","""pbp_12027665.parquet""",12027665,1,4,9,6
"""20240203""","""pbp_12027669.parquet""",12027669,1,4,12,9
…,…,…,…,…,…,…
"""20240305""","""pbp_12126012.parquet""",12126012,2,4,10,7
"""20240305""","""pbp_12126659.parquet""",12126659,2,4,8,5
"""20240320""","""pbp_12156279.parquet""",12156279,3,2,7,6


In [22]:
# Show only the sets that actually failed the game_id continuity check and preserve their exact game IDs for inspection

invalid_game_id_details = []

for file, check in invalid_game_sequences:
    bad_sets = check.select(["match_id", "set_id"])

    details = (
        pl.read_parquet(file)
        .join(
            bad_sets,
            on=["match_id", "set_id"],
            how="inner"
        )
        .group_by(["match_id", "set_id"])
        .agg(
            pl.col("game_id")
            .unique()
            .sort()
            .alias("game_ids")
        )
        .with_columns(
            pl.lit(file.name).alias("file_name"),
            pl.lit(file.parent.parent.name).alias("snapshot_date")
        )
    )

    invalid_game_id_details.append(details)

invalid_game_id_details_df = (
    pl.concat(invalid_game_id_details)
    .sort(["snapshot_date", "match_id", "set_id"])
)


for row in invalid_game_id_details_df.iter_rows(named=True):
    print(
        row["snapshot_date"],
        "|",
        row["file_name"],
        "| set:",
        row["set_id"],
        "| games:",
        row["game_ids"]
    )

20240202 | pbp_12027665.parquet | set: 1 | games: [4, 5, 6, 7, 8, 9]
20240202 | pbp_12027669.parquet | set: 1 | games: [4, 5, 6, 7, 8, 9, 10, 11, 12]
20240202 | pbp_12027679.parquet | set: 1 | games: [1, 2, 3, 5, 6, 7]
20240203 | pbp_12027665.parquet | set: 1 | games: [4, 5, 6, 7, 8, 9]
20240203 | pbp_12027669.parquet | set: 1 | games: [4, 5, 6, 7, 8, 9, 10, 11, 12]
20240203 | pbp_12027679.parquet | set: 1 | games: [1, 2, 3, 5, 6, 7]
20240219 | pbp_12083047.parquet | set: 1 | games: [1, 3, 4, 5, 6, 7, 8]
20240220 | pbp_12083047.parquet | set: 1 | games: [1, 3, 4, 5, 6, 7, 8]
20240304 | pbp_12126012.parquet | set: 2 | games: [4, 5, 6, 7, 8, 9, 10]
20240304 | pbp_12126659.parquet | set: 2 | games: [4, 5, 6, 7, 8]
20240305 | pbp_12126012.parquet | set: 2 | games: [4, 5, 6, 7, 8, 9, 10]
20240305 | pbp_12126659.parquet | set: 2 | games: [4, 5, 6, 7, 8]
20240320 | pbp_12156279.parquet | set: 3 | games: [2, 3, 4, 5, 6, 7]
20240321 | pbp_12156279.parquet | set: 3 | games: [2, 3, 4, 5, 6, 7]
20

### 4.7 Investigate internal missing games

Most failed game sequences represent partial sets whose point-by-point coverage begins after game 1.

Two unique matches contain an internal missing game, so their surrounding set scores are inspected before deciding how they should be handled.

In [23]:
# Inspect game-level scores around the two internal gaps
# This helps determine whether a game occurred but its point rows are missing

internal_gap_matches = [12027679, 12083047]

for match_id in internal_gap_matches:
    print(f"\nMatch {match_id}")

    # Use the first available file appearance for inspection
    file = next(
        file for file in point_files
        if file.stem == f"pbp_{match_id}"
    )

    df = pl.read_parquet(file)

    summary = (
        df.group_by(["set_id", "game_id"])
        .agg(
            pl.col("home_score").first().alias("home_score"),
            pl.col("away_score").first().alias("away_score"),
            pl.len().alias("point_rows")
        )
        .sort(["set_id", "game_id"])
    )

    print(summary)


Match 12027679
shape: (23, 5)
┌────────┬─────────┬────────────┬────────────┬────────────┐
│ set_id ┆ game_id ┆ home_score ┆ away_score ┆ point_rows │
│ ---    ┆ ---     ┆ ---        ┆ ---        ┆ ---        │
│ i64    ┆ i64     ┆ i64        ┆ i64        ┆ u32        │
╞════════╪═════════╪════════════╪════════════╪════════════╡
│ 1      ┆ 1       ┆ 0          ┆ 1          ┆ 4          │
│ 1      ┆ 2       ┆ 1          ┆ 1          ┆ 7          │
│ 1      ┆ 3       ┆ 1          ┆ 2          ┆ 22         │
│ 1      ┆ 5       ┆ 1          ┆ 4          ┆ 4          │
│ 1      ┆ 6       ┆ 1          ┆ 5          ┆ 5          │
│ …      ┆ …       ┆ …          ┆ …          ┆ …          │
│ 3      ┆ 3       ┆ 2          ┆ 1          ┆ 11         │
│ 3      ┆ 4       ┆ 3          ┆ 1          ┆ 7          │
│ 3      ┆ 5       ┆ 4          ┆ 1          ┆ 7          │
│ 3      ┆ 6       ┆ 5          ┆ 1          ┆ 3          │
│ 3      ┆ 7       ┆ 6          ┆ 1          ┆ 4          │
└────────

In [24]:
# Inspect only the games around the internal gap in match 12083047

match_id = 12083047

file = next(
    file for file in point_files
    if file.stem == f"pbp_{match_id}"
)

df = pl.read_parquet(file)

(
    df.filter(
        (pl.col("set_id") == 1)
        & (pl.col("game_id") <= 4)
    )
    .group_by(["set_id", "game_id"])
    .agg(
        pl.col("home_score").first().alias("home_score"),
        pl.col("away_score").first().alias("away_score"),
        pl.len().alias("point_rows")
    )
    .sort("game_id")
)

set_id,game_id,home_score,away_score,point_rows
i64,i64,i64,i64,u32
1,1,1,0,5
1,3,2,1,5
1,4,2,2,5


The internal gaps were confirmed as incomplete point-by-point records.

- Match `12027679` is missing game 4 of set 1.
- Match `12083047` is missing game 2 of set 1.

The surrounding set scores show that these games occurred, but their point-by-point rows are absent.

Missing games are not reconstructed because doing so would require guessing unavailable point-level information.

In [25]:
# Record incomplete point-by-point snapshots without removing any data
# Partial-start sequences and internal gaps are kept as separate issue types

incomplete_pbp_flags = pl.DataFrame(
    {
        "snapshot_date": [
            "20240202", "20240202", "20240202",
            "20240203", "20240203", "20240203",
            "20240219", "20240220",
            "20240304", "20240304",
            "20240305", "20240305",
            "20240320", "20240321", "20240322"
        ],
        "match_id": [
            12027665, 12027669, 12027679,
            12027665, 12027669, 12027679,
            12083047, 12083047,
            12126012, 12126659,
            12126012, 12126659,
            12156279, 12156279, 12156279
        ],
        "sequence_issue": [
            "partial_set_start",
            "partial_set_start",
            "internal_missing_game",
            "partial_set_start",
            "partial_set_start",
            "internal_missing_game",
            "internal_missing_game",
            "internal_missing_game",
            "partial_set_start",
            "partial_set_start",
            "partial_set_start",
            "partial_set_start",
            "partial_set_start",
            "partial_set_start",
            "partial_set_start"
        ]
    }
)

incomplete_pbp_flags

snapshot_date,match_id,sequence_issue
str,i64,str
"""20240202""",12027665,"""partial_set_start"""
"""20240202""",12027669,"""partial_set_start"""
"""20240202""",12027679,"""internal_missing_game"""
"""20240203""",12027665,"""partial_set_start"""
"""20240203""",12027669,"""partial_set_start"""
…,…,…
"""20240305""",12126012,"""partial_set_start"""
"""20240305""",12126659,"""partial_set_start"""
"""20240320""",12156279,"""partial_set_start"""


Seven unique matches contain incomplete point-by-point coverage across 15 daily snapshots.

Five matches begin partway through a set, while two matches contain an internal missing game.

These observations are flagged rather than removed or reconstructed. The flags will later prevent incomplete sequences from being incorrectly classified as violations of tennis scoring rules.

### 4.8 Validate set numbering across the complete dataset

In [26]:
# Check set_id structure in every point-by-point file
# We distinguish a match that starts at a later set from one with a missing set internally

set_sequence_issues = []

for file in point_files:
    df = pl.read_parquet(file)

    set_ids = (
        df.select("set_id")
        .unique()
        .sort("set_id")
        .get_column("set_id")
        .to_list()
    )

    min_set = min(set_ids)
    max_set = max(set_ids)

    expected_sets = list(range(min_set, max_set + 1))

    if min_set != 1 or set_ids != expected_sets:
        set_sequence_issues.append(
            {
                "snapshot_date": file.parent.parent.name,
                "file_name": file.name,
                "match_id": df["match_id"][0],
                "set_ids": set_ids,
                "starts_after_set_1": min_set != 1,
                "internal_set_gap": set_ids != expected_sets,
            }
        )

print("Files with set sequence issues:", len(set_sequence_issues))

Files with set sequence issues: 0


### Stage 4 findings

Structural validation of the point-by-point data showed that:

- 22,255 of 22,272 raw files are not stored in chronological row order, so the combined dataset must be sorted by `snapshot_date -> match_id -> set_id -> game_id -> point_id`.
- No invalid `point_id` sequences were found. Within the recorded portion of each game, point IDs start at 0 and are unique, consecutive, and gap-free. This does not by itself guarantee that the end of every game was recorded.
- 15 file appearances contain incomplete game sequences, representing 7 unique matches.
- Five unique matches contain partial set coverage where point-by-point recording begins after the start of a set.
- Two unique matches contain an internal missing game confirmed by surrounding set scores.
- These incomplete sequences are flagged rather than deleted or reconstructed.
- No missing or discontinuous `set_id` sequences were found across the complete dataset.

Incomplete point-by-point coverage is treated as missing data, not automatically as a violation of tennis rules.

## 5. Investigate repeated daily snapshots

### 5.1 Count repeated point-by-point filenames

In [27]:
# Count how often each point-by-point filename appears across daily snapshots
# The same match file may appear on several different snapshot dates

file_name_counts = {}

for file in point_files:
    file_name_counts[file.name] = file_name_counts.get(file.name, 0) + 1

unique_file_names = len(file_name_counts)

repeated_file_names = sum(
    count > 1
    for count in file_name_counts.values()
)

print("Total file appearances:", len(point_files))
print("Unique filenames:", unique_file_names)
print("Repeated filenames:", repeated_file_names)

Total file appearances: 22272
Unique filenames: 10956
Repeated filenames: 10129


The point-by-point dataset contains 22,272 file appearances across all daily snapshots.

There are 10,956 unique filenames, of which 10,129 appear on more than one snapshot date.

This means most match files are observed repeatedly across multiple daily snapshots, so repeated filenames must be compared by content rather than treated as accidental duplicates.

### 5.2 Check whether repeated files change across snapshots

In [28]:
# Compare repeated files after normalizing row order
# This prevents row-order differences from being mistaken for content changes

first_versions = {}
changed_files = set()

for file in point_files:
    name = file.name

    if file_name_counts[name] <= 1:
        continue

    df = (
        pl.read_parquet(file)
        .sort(["set_id", "game_id", "point_id"])
    )

    if name not in first_versions:
        first_versions[name] = df

    elif not df.equals(first_versions[name]):
        changed_files.add(name)

unchanged_repeated_files = {
    name
    for name, count in file_name_counts.items()
    if count > 1
} - changed_files

print("Repeated filenames:", repeated_file_names)
print("Changed repeated filenames:", len(changed_files))
print("Unchanged repeated filenames:", len(unchanged_repeated_files))

Repeated filenames: 10129
Changed repeated filenames: 59
Unchanged repeated filenames: 10070


Most repeated point-by-point filenames remain logically unchanged across snapshot dates.

Out of 10,129 repeated filenames:

- 59 changed in content across at least one snapshot.
- 10,070 remained unchanged across all observed snapshots.

Repeated files are still preserved because each appearance belongs to a different `snapshot_date`.

### 5.3 Inspect one changed point-by-point file

In [29]:
# Inspect one repeated filename whose point-by-point content changed across snapshots

changed_file_name = sorted(changed_files)[0]

changed_versions = []

for file in point_files:
    if file.name == changed_file_name:
        df = (
            pl.read_parquet(file)
            .with_columns(
                pl.lit(file.parent.parent.name).alias("snapshot_date")
            )
        )

        changed_versions.append(df)

print("Changed file:", changed_file_name)
print("Number of snapshots:", len(changed_versions))

Changed file: pbp_12060690.parquet
Number of snapshots: 2


### 5.4 Classify how repeated files changed

In [30]:
# Compare row counts across snapshots for every genuinely changed filename
# This distinguishes growing/partial records from same-size content revisions

row_count_change = []
same_size_change = []

for name in changed_files:
    versions = []

    for file in point_files:
        if file.name == name:
            df = pl.read_parquet(file)
            versions.append(df.height)

    if len(set(versions)) > 1:
        row_count_change.append(name)
    else:
        same_size_change.append(name)

print("Changed files with different row counts:", len(row_count_change))
print("Changed files with the same row count:", len(same_size_change))

Changed files with different row counts: 59
Changed files with the same row count: 0


### 5.5 Check whether changed snapshots are cumulative

In [31]:
# Check whether changed point-by-point snapshots grow cumulatively
# We verify that rows from an earlier snapshot remain present in the next snapshot

cumulative_changes = 0
non_cumulative_changes = []

for name in changed_files:
    versions = []

    for file in point_files:
        if file.name == name:
            df = (
                pl.read_parquet(file)
                .sort(["set_id", "game_id", "point_id"])
            )

            versions.append(
                (file.parent.parent.name, df)
            )

    versions.sort(key=lambda x: x[0])

    file_is_cumulative = True

    for i in range(len(versions) - 1):
        old_date, old_df = versions[i]
        new_date, new_df = versions[i + 1]

        missing_from_new = old_df.join(
            new_df,
            on=old_df.columns,
            how="anti"
        )

        if missing_from_new.height > 0:
            file_is_cumulative = False
            non_cumulative_changes.append(
                (name, old_date, new_date, missing_from_new.height)
            )

    if file_is_cumulative:
        cumulative_changes += 1

print("Changed filenames with cumulative growth:", cumulative_changes)
print("Changed filenames with non-cumulative changes:", len(set(x[0] for x in non_cumulative_changes)))

Changed filenames with cumulative growth: 58
Changed filenames with non-cumulative changes: 1


### 5.6 Investigate the non-cumulative snapshot change

In [32]:
# Identify the changed file whose later snapshot does not fully preserve the rows that were present in the earlier snapshot

for item in non_cumulative_changes:
    print(item)

('pbp_12075951.parquet', '20240217', '20240218', 5)


In [33]:
# Load the two snapshot versions of the non-cumulative file

old_file = next(
    file for file in point_files
    if file.name == "pbp_12075951.parquet"
    and file.parent.parent.name == "20240217"
)

new_file = next(
    file for file in point_files
    if file.name == "pbp_12075951.parquet"
    and file.parent.parent.name == "20240218"
)

old_df = pl.read_parquet(old_file).sort(
    ["set_id", "game_id", "point_id"]
)

new_df = pl.read_parquet(new_file).sort(
    ["set_id", "game_id", "point_id"]
)

In [34]:
# Show the five rows from the earlier snapshot that are not identical in the later snapshot

old_rows_not_preserved = old_df.join(
    new_df,
    on=old_df.columns,
    how="anti"
)

old_rows_not_preserved

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
12075951,2,1,0,"""15""","""0""",0,1,5,1,0,2,1
12075951,2,1,1,"""30""","""0""",0,1,5,1,0,2,1
12075951,2,1,2,"""30""","""15""",0,5,1,1,0,2,1
12075951,2,1,3,"""40""","""15""",2,2,5,1,0,2,1
12075951,2,1,4,"""40""","""30""",0,2,1,1,0,2,1


In [35]:
# Inspect the same point identifiers in the later snapshot
# This tells us whether the rows were deleted or revised

new_matching_rows = (
    new_df
    .filter(
        (pl.col("set_id") == 2)
        & (pl.col("game_id") == 1)
        & (pl.col("point_id").is_between(0, 4))
    )
    .sort("point_id")
)

new_matching_rows

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring
i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
12075951,2,1,0,"""30""","""0""",0,1,5,1,0,2,1
12075951,2,1,1,"""30""","""15""",0,5,1,1,0,2,1
12075951,2,1,2,"""40""","""15""",2,2,5,1,0,2,1
12075951,2,1,3,"""40""","""30""",0,2,1,1,0,2,1


In [36]:
# Record the snapshot where previously observed point-by-point data was revised
# The raw data is preserved; this flag only documents the detected revision

snapshot_revision_flags = pl.DataFrame(
    {
        "snapshot_date": ["20240218"],
        "match_id": [12075951],
        "snapshot_issue": ["snapshot_revision"]
    }
)

snapshot_revision_flags

snapshot_date,match_id,snapshot_issue
str,i64,str
"""20240218""",12075951,"""snapshot_revision"""


The only non-cumulative repeated file was `pbp_12075951.parquet`.

Between the 2024-02-17 and 2024-02-18 snapshots, one point row (`15-0`) disappeared from set 2, game 1, and the remaining point IDs were renumbered.

This represents a genuine revision of previously recorded point-by-point data rather than simple cumulative growth.

Both snapshots are preserved because they represent the data available on different dates. No attempt is made to reconstruct or overwrite the historical version.

## 6. Build the full point-by-point dataset

### 6.1 Combine all daily point-by-point snapshots

In [37]:
# Read all point-by-point files into one table while preserving each source path
# The source path is used to recover the daily snapshot date

point_all = pl.read_parquet(
    [str(file) for file in point_files],
    include_file_paths="source_file"
)

point_all.shape

(2549369, 14)

### 6.2 Add the daily snapshot date

In [38]:
# Extract the snapshot date from each source file path
# The date keeps repeated match observations from different days distinct

point_all = (
    point_all
    .with_columns(
        pl.col("source_file")
        .str.extract(r"extracted[\\/](\d{8})", 1)
        .str.to_date("%Y%m%d")
        .alias("snapshot_date")
    )
    .drop("source_file")
)

point_all.select("snapshot_date").head()

snapshot_date
date
2024-02-01
2024-02-01
2024-02-01
2024-02-01
2024-02-01


In [39]:
# Confirm that every row received a valid snapshot date

point_all.select(
    pl.col("snapshot_date").null_count().alias("missing_snapshot_dates")
)

missing_snapshot_dates
u32
0


### 6.3 Sort the complete dataset into chronological order

In [40]:
# Sort each daily match snapshot into tennis sequence order
# Different snapshot dates are kept separate

point_all = point_all.sort(
    [
        "snapshot_date",
        "match_id",
        "set_id",
        "game_id",
        "point_id"
    ]
)

point_all.head(20)

match_id,set_id,game_id,point_id,home_point,away_point,point_description,home_point_type,away_point_type,home_score,away_score,serving,scoring,snapshot_date
i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,date
11998445,1,1,0,"""0""","""15""",0,5,1,0,1,2,2,2024-02-01
11998445,1,1,1,"""0""","""30""",0,5,1,0,1,2,2,2024-02-01
11998445,1,1,2,"""0""","""40""",0,5,1,0,1,2,2,2024-02-01
11998445,1,2,0,"""15""","""0""",0,1,5,1,1,1,1,2024-02-01
11998445,1,2,1,"""30""","""0""",0,1,5,1,1,1,1,2024-02-01
…,…,…,…,…,…,…,…,…,…,…,…,…,…
11998445,1,5,1,"""15""","""15""",0,5,1,3,2,2,1,2024-02-01
11998445,1,5,2,"""30""","""15""",0,1,5,3,2,2,1,2024-02-01
11998445,1,5,3,"""40""","""15""",0,2,5,3,2,2,1,2024-02-01


In [41]:
# Confirm that the complete combined table is now stored in the intended order

sort_columns = [
    "snapshot_date",
    "match_id",
    "set_id",
    "game_id",
    "point_id"
]

print(
    "Full dataset sorted correctly:",
    point_all.select(sort_columns).equals(
        point_all.sort(sort_columns).select(sort_columns)
    )
)

Full dataset sorted correctly: True


### 6.4 Attach structural data-quality flags

In [42]:
# Convert flag dates to the same Date type used in the full dataset

incomplete_pbp_flags = incomplete_pbp_flags.with_columns(
    pl.col("snapshot_date")
    .str.to_date("%Y%m%d")
)

snapshot_revision_flags = snapshot_revision_flags.with_columns(
    pl.col("snapshot_date")
    .str.to_date("%Y%m%d")
)

In [43]:
# Attach incomplete-sequence information to the corresponding match snapshots

point_all = point_all.join(
    incomplete_pbp_flags,
    on=["snapshot_date", "match_id"],
    how="left"
)

In [44]:
# Attach the snapshot-revision flag identified during repeated-snapshot analysis

point_all = point_all.join(
    snapshot_revision_flags,
    on=["snapshot_date", "match_id"],
    how="left"
)

In [45]:
point_all.select(
    [
        "snapshot_date",
        "match_id",
        "sequence_issue",
        "snapshot_issue"
    ]
).filter(
    pl.col("sequence_issue").is_not_null()
    | pl.col("snapshot_issue").is_not_null()
).unique().sort(
    ["snapshot_date", "match_id"]
)

snapshot_date,match_id,sequence_issue,snapshot_issue
date,i64,str,str
2024-02-02,12027665,"""partial_set_start""",null
2024-02-02,12027669,"""partial_set_start""",null
2024-02-02,12027679,"""internal_missing_game""",null
2024-02-03,12027665,"""partial_set_start""",null
2024-02-03,12027669,"""partial_set_start""",null
…,…,…,…
2024-03-05,12126012,"""partial_set_start""",null
2024-03-05,12126659,"""partial_set_start""",null
2024-03-20,12156279,"""partial_set_start""",null


## 7. Validate tennis scoring rules

### 7.1 Inspect the point-score values used in the dataset

In [46]:
# Collect every distinct point-score value used by either player
# We inspect the actual encoding before defining tennis-rule validation

point_values = (
    pl.concat(
        [
            point_all.select(
                pl.col("home_point").alias("point_value")
            ),
            point_all.select(
                pl.col("away_point").alias("point_value")
            )
        ]
    )
    .group_by("point_value")
    .len()
    .sort("point_value")
)

pl.Config.set_tbl_rows(100)
point_values

point_value,len
str,u32
"""0""",844020
"""1""",25931
"""10""",1825
"""11""",640
"""12""",309
"""13""",145
"""14""",71
"""15""",1439462
"""16""",14


The dataset uses both standard tennis point notation (`0`, `15`, `30`, `40`, `A`) and numeric point notation (`1`, `2`, `3`, ...).

Numeric values above the standard point-score set are not treated as invalid automatically because they may represent tie-break or match tie-break scoring. Games must therefore be classified by their complete point sequence and match context before tennis-rule validation.

### 7.2 Identify games that use numeric tie-break-style scoring

In [47]:
# Identify games containing point values outside normal 0-15-30-40-A scoring
# These are candidates for tie-break or other numeric scoring formats

normal_point_values = ["0", "15", "30", "40", "A"]

numeric_scoring_games = (
    point_all
    .filter(
        (~pl.col("home_point").is_in(normal_point_values))
        |
        (~pl.col("away_point").is_in(normal_point_values))
    )
    .select(
        [
            "snapshot_date",
            "match_id",
            "set_id",
            "game_id"
        ]
    )
    .unique()
)

print(
    "Games using numeric point scoring:",
    numeric_scoring_games.height
)

Games using numeric point scoring: 6433


### 7.3 Inspect the set-score context of numeric-scoring games

In [48]:
# Summarize the set score associated with each numeric-scoring game
# This helps determine whether these games occur in expected tie-break situations

game_score_summary = (
    point_all
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"]
    )
    .agg(
        pl.col("home_score").first().alias("home_score"),
        pl.col("away_score").first().alias("away_score")
    )
)

numeric_game_scores = (
    numeric_scoring_games
    .join(
        game_score_summary,
        on=["snapshot_date", "match_id", "set_id", "game_id"],
        how="left"
    )
)

(
    numeric_game_scores
    .group_by(["home_score", "away_score"])
    .len()
    .sort("len", descending=True)
)

home_score,away_score,len
i64,i64,u32
6,7,2758
7,6,2743
6,10,99
8,10,84
10,6,74
10,7,72
7,10,70
10,4,57
10,5,56


The numeric-scoring games appear to contain more than one tennis scoring format.

Most numeric games end with a set score of `7-6` or `6-7`, which is consistent with a standard set tie-break.

Other numeric games contain scores such as `10-6`, `11-9`, and `21-19`. These are consistent with match tie-break or super tie-break formats, where numeric scoring continues until one player reaches the target score with the required winning margin.

Numeric scoring is therefore not treated as invalid solely because it differs from normal `0-15-30-40-A` scoring.

### 7.4 Verify game-level score consistency

In [49]:
# Verify that home_score and away_score remain constant within each recorded game
# This confirms that they can safely be used as the game's set-score context

inconsistent_game_scores = (
    point_all
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"]
    )
    .agg(
        pl.col("home_score").n_unique().alias("home_score_values"),
        pl.col("away_score").n_unique().alias("away_score_values")
    )
    .filter(
        (pl.col("home_score_values") > 1)
        |
        (pl.col("away_score_values") > 1)
    )
)

print(
    "Games with changing home_score/away_score:",
    inconsistent_game_scores.height
)

Games with changing home_score/away_score: 0


### 7.5 Investigate non-standard numeric-scoring games

In [50]:
# Separate standard 7-6 / 6-7 tie-break candidates from other games that use numeric point scoring

other_numeric_games = (
    numeric_game_scores
    .filter(
        ~(
            ((pl.col("home_score") == 7) & (pl.col("away_score") == 6))
            |
            ((pl.col("home_score") == 6) & (pl.col("away_score") == 7))
        )
    )
)

print("Standard 7-6 / 6-7 tie-break candidates:",
      numeric_game_scores.height - other_numeric_games.height)

print("Other numeric-scoring games:",
      other_numeric_games.height)

Standard 7-6 / 6-7 tie-break candidates: 5501
Other numeric-scoring games: 932


In [51]:
# Inspect which set and game numbers contain the other numeric scoring formats

(
    other_numeric_games
    .group_by(["set_id", "game_id"])
    .len()
    .sort("len", descending=True)
)

set_id,game_id,len
i64,i64,u32
3,1,932


All 932 non-standard numeric-scoring games occur in `set_id = 3` and `game_id = 1`.

This strongly suggests that the dataset represents deciding match tie-breaks as a third set containing a single numeric-scoring game.

These observations are therefore treated separately from standard set tie-breaks rather than being classified as invalid tennis scores.

### 7.6 Validate deciding match tie-break results

In [52]:
# Check the final scores of deciding match tie-break candidates
# A match tie-break should normally finish with at least 10 points and a winning margin of at least 2 points

match_tiebreak_check = (
    other_numeric_games
    .with_columns(
        pl.max_horizontal(
            "home_score",
            "away_score"
        ).alias("winner_score"),

        (
            pl.col("home_score") - pl.col("away_score")
        ).abs().alias("score_margin")
    )
    .with_columns(
        (
            (pl.col("winner_score") >= 10)
            &
            (pl.col("score_margin") >= 2)
        ).alias("valid_match_tiebreak_result")
    )
)

match_tiebreak_check.filter(
    ~pl.col("valid_match_tiebreak_result")
)

snapshot_date,match_id,set_id,game_id,home_score,away_score,winner_score,score_margin,valid_match_tiebreak_result
date,i64,i64,i64,i64,i64,i64,i64,bool
2024-02-05,12041844,3,1,1,0,1,1,false
2024-03-25,12190437,3,1,0,1,1,1,false
2024-02-04,12040872,3,1,0,1,1,1,false
2024-02-03,12040872,3,1,0,1,1,1,false
2024-02-04,12041844,3,1,1,0,1,1,false
2024-03-24,12190437,3,1,0,1,1,1,false


### 7.7 Investigate unusual deciding-set score encoding

A small number of deciding-set numeric games have `home_score` / `away_score` values of `1-0` or `0-1`.

These observations are inspected separately because the game-level score fields may use a different encoding from the actual numeric point progression.

In [53]:
# Inspect the final numeric point score for the unusual deciding-set games

unusual_match_tiebreaks = (
    match_tiebreak_check
    .filter(~pl.col("valid_match_tiebreak_result"))
    .select(
        ["snapshot_date", "match_id", "set_id", "game_id"]
    )
)

unusual_tiebreak_summary = (
    point_all
    .join(
        unusual_match_tiebreaks,
        on=["snapshot_date", "match_id", "set_id", "game_id"],
        how="inner"
    )
    .sort(
        ["snapshot_date", "match_id", "point_id"]
    )
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"],
        maintain_order=True
    )
    .agg(
        pl.len().alias("point_rows"),
        pl.col("point_id").max().alias("max_point_id"),
        pl.col("home_point").last().alias("final_home_point"),
        pl.col("away_point").last().alias("final_away_point")
    )
)

unusual_tiebreak_summary

snapshot_date,match_id,set_id,game_id,point_rows,max_point_id,final_home_point,final_away_point
date,i64,i64,i64,u32,i64,str,str
2024-02-03,12040872,3,1,16,15,"""6""","""10"""
2024-02-04,12040872,3,1,16,15,"""6""","""10"""
2024-02-04,12041844,3,1,18,17,"""10""","""8"""
2024-02-05,12041844,3,1,18,17,"""10""","""8"""
2024-03-24,12190437,3,1,28,27,"""13""","""15"""
2024-03-25,12190437,3,1,28,27,"""13""","""15"""


The unusual `1-0` and `0-1` game-level scores were not tennis-rule violations.

Inspection of the actual numeric point progression showed valid deciding match tie-break results of `10-6`, `10-8`, and `15-13`.

For deciding match tie-breaks, final `home_point` and `away_point` values are therefore used for rule validation rather than `home_score` and `away_score`.

### 7.8 Validate all deciding match tie-breaks using final point scores

In [54]:
# Get the final point score of every deciding match tie-break candidate

match_tiebreak_results = (
    point_all
    .join(
        other_numeric_games.select(
            ["snapshot_date", "match_id", "set_id", "game_id"]
        ),
        on=["snapshot_date", "match_id", "set_id", "game_id"],
        how="inner"
    )
    .sort(
        ["snapshot_date", "match_id", "set_id", "game_id", "point_id"]
    )
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"],
        maintain_order=True
    )
    .agg(
        pl.col("home_point")
        .last()
        .cast(pl.Int64)
        .alias("final_home_point"),

        pl.col("away_point")
        .last()
        .cast(pl.Int64)
        .alias("final_away_point")
    )
    .with_columns(
        pl.max_horizontal(
            "final_home_point",
            "final_away_point"
        ).alias("winner_points"),

        (
            pl.col("final_home_point")
            - pl.col("final_away_point")
        ).abs().alias("winning_margin")
    )
    .with_columns(
        (
            (pl.col("winner_points") >= 10)
            &
            (pl.col("winning_margin") >= 2)
        ).alias("valid_match_tiebreak")
    )
)

match_tiebreak_results.filter(
    ~pl.col("valid_match_tiebreak")
)

snapshot_date,match_id,set_id,game_id,final_home_point,final_away_point,winner_points,winning_margin,valid_match_tiebreak
date,i64,i64,i64,i64,i64,i64,i64,bool


All 932 deciding match tie-break observations satisfy the expected ending rule when validated using their final numeric point scores.

Each winner reached at least 10 points and won by a margin of at least 2 points.

No deciding match tie-break observations were classified as invalid.

### 7.9 Validate standard set tie-break results

In [55]:
# Isolate standard set tie-break candidates
# These are numeric-scoring games whose set score is 7-6 or 6-7

standard_tiebreak_games = (
    numeric_game_scores
    .filter(
        (
            (pl.col("home_score") == 7)
            & (pl.col("away_score") == 6)
        )
        |
        (
            (pl.col("home_score") == 6)
            & (pl.col("away_score") == 7)
        )
    )
    .select(
        ["snapshot_date", "match_id", "set_id", "game_id"]
    )
)

standard_tiebreak_results = (
    point_all
    .join(
        standard_tiebreak_games,
        on=["snapshot_date", "match_id", "set_id", "game_id"],
        how="inner"
    )
    .sort(
        ["snapshot_date", "match_id", "set_id", "game_id", "point_id"]
    )
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"],
        maintain_order=True
    )
    .agg(
        pl.col("home_point")
        .last()
        .cast(pl.Int64)
        .alias("final_home_point"),

        pl.col("away_point")
        .last()
        .cast(pl.Int64)
        .alias("final_away_point")
    )
    .with_columns(
        pl.max_horizontal(
            "final_home_point",
            "final_away_point"
        ).alias("winner_points"),

        (
            pl.col("final_home_point")
            - pl.col("final_away_point")
        ).abs().alias("winning_margin")
    )
    .with_columns(
        (
            (pl.col("winner_points") >= 7)
            &
            (pl.col("winning_margin") >= 2)
        ).alias("valid_standard_tiebreak")
    )
)

standard_tiebreak_results.filter(
    ~pl.col("valid_standard_tiebreak")
)

snapshot_date,match_id,set_id,game_id,final_home_point,final_away_point,winner_points,winning_margin,valid_standard_tiebreak
date,i64,i64,i64,i64,i64,i64,i64,bool
2024-02-01,11999211,3,13,4,6,6,2,false
2024-02-02,11999211,3,13,4,6,6,2,false
2024-02-27,12088042,2,13,5,4,5,1,false
2024-02-27,12100263,2,13,6,3,6,3,false
2024-02-28,12088042,2,13,5,4,5,1,false
2024-02-28,12100263,2,13,6,3,6,3,false
2024-03-22,12184203,3,13,6,5,6,1,false
2024-03-25,12159545,1,13,6,4,6,2,false
2024-03-26,12159545,1,13,6,4,6,2,false


### 7.10 Investigate incomplete-looking standard tie-breaks

In [56]:
# Inspect every recorded point for the standard tie-breaks that did not satisfy the expected final-score rule

suspicious_standard_tiebreaks = (
    standard_tiebreak_results
    .filter(~pl.col("valid_standard_tiebreak"))
    .select(
        ["snapshot_date", "match_id", "set_id", "game_id"]
    )
)

suspicious_tiebreak_points = (
    point_all
    .join(
        suspicious_standard_tiebreaks,
        on=["snapshot_date", "match_id", "set_id", "game_id"],
        how="inner"
    )
    .select(
        [
            "snapshot_date",
            "match_id",
            "set_id",
            "game_id",
            "point_id",
            "home_point",
            "away_point",
            "home_score",
            "away_score"
        ]
    )
    .sort(
        ["snapshot_date", "match_id", "set_id", "game_id", "point_id"]
    )
)

suspicious_tiebreak_points

snapshot_date,match_id,set_id,game_id,point_id,home_point,away_point,home_score,away_score
date,i64,i64,i64,i64,str,str,i64,i64
2024-02-01,11999211,3,13,0,"""1""","""0""",6,7
2024-02-01,11999211,3,13,1,"""2""","""0""",6,7
2024-02-01,11999211,3,13,2,"""2""","""1""",6,7
2024-02-01,11999211,3,13,3,"""2""","""2""",6,7
2024-02-01,11999211,3,13,4,"""2""","""3""",6,7
2024-02-01,11999211,3,13,5,"""3""","""3""",6,7
2024-02-01,11999211,3,13,6,"""3""","""4""",6,7
2024-02-01,11999211,3,13,7,"""3""","""5""",6,7
2024-02-01,11999211,3,13,8,"""3""","""6""",6,7


The nine standard tie-break observations that failed the final-score check are incomplete point-by-point records rather than invalid tennis results.

Their recorded point sequences progress normally but stop before the tie-break reaches a legal winning score, while `home_score` and `away_score` already indicate that the set finished `7-6` or `6-7`.

These observations are retained and flagged as incomplete tie-break point-by-point data. Missing points are not reconstructed.

In [57]:
# Flag standard tie-break snapshots whose point-by-point sequence stops before the legal end of the tie-break

incomplete_tiebreak_flags = (
    standard_tiebreak_results
    .filter(~pl.col("valid_standard_tiebreak"))
    .select(
        ["snapshot_date", "match_id"]
    )
    .with_columns(
        pl.lit("incomplete_tiebreak_end")
        .alias("tiebreak_issue")
    )
)

incomplete_tiebreak_flags

snapshot_date,match_id,tiebreak_issue
date,i64,str
2024-02-01,11999211,"""incomplete_tiebreak_end"""
2024-02-02,11999211,"""incomplete_tiebreak_end"""
2024-02-27,12088042,"""incomplete_tiebreak_end"""
2024-02-27,12100263,"""incomplete_tiebreak_end"""
2024-02-28,12088042,"""incomplete_tiebreak_end"""
2024-02-28,12100263,"""incomplete_tiebreak_end"""
2024-03-22,12184203,"""incomplete_tiebreak_end"""
2024-03-25,12159545,"""incomplete_tiebreak_end"""
2024-03-26,12159545,"""incomplete_tiebreak_end"""


### 7.11 Attach incomplete tie-break flags

In [58]:
# Attach the incomplete tie-break flag without modifying the structural sequence and snapshot-revision flags

rows_before_tiebreak_join = point_all.height

point_all = point_all.join(
    incomplete_tiebreak_flags,
    on=["snapshot_date", "match_id"],
    how="left"
)

print("Rows before join:", rows_before_tiebreak_join)
print("Rows after join:", point_all.height)
print(
    "Row count preserved:",
    rows_before_tiebreak_join == point_all.height
)

Rows before join: 2549369
Rows after join: 2549369
Row count preserved: True


In [59]:
(
    point_all
    .filter(pl.col("tiebreak_issue").is_not_null())
    .select(
        ["snapshot_date", "match_id", "tiebreak_issue"]
    )
    .unique()
    .sort(["snapshot_date", "match_id"])
)

snapshot_date,match_id,tiebreak_issue
date,i64,str
2024-02-01,11999211,"""incomplete_tiebreak_end"""
2024-02-02,11999211,"""incomplete_tiebreak_end"""
2024-02-27,12088042,"""incomplete_tiebreak_end"""
2024-02-27,12100263,"""incomplete_tiebreak_end"""
2024-02-28,12088042,"""incomplete_tiebreak_end"""
2024-02-28,12100263,"""incomplete_tiebreak_end"""
2024-03-22,12184203,"""incomplete_tiebreak_end"""
2024-03-25,12159545,"""incomplete_tiebreak_end"""
2024-03-26,12159545,"""incomplete_tiebreak_end"""


### 7.12 Inspect how standard tennis games are recorded at completion

In [60]:
# Games that never use numeric tie-break scoring
standard_games = (
    point_all
    .join(
        numeric_scoring_games,
        on=["snapshot_date", "match_id", "set_id", "game_id"],
        how="anti"
    )
)

# Inspect the final recorded point score of each standard game
standard_game_endings = (
    standard_games
    .sort(
        ["snapshot_date", "match_id", "set_id", "game_id", "point_id"]
    )
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"],
        maintain_order=True
    )
    .agg(
        pl.col("home_point").last().alias("final_home_point"),
        pl.col("away_point").last().alias("final_away_point"),
        pl.len().alias("point_rows")
    )
)

(
    standard_game_endings
    .group_by(["final_home_point", "final_away_point"])
    .len()
    .sort("len", descending=True)
)

final_home_point,final_away_point,len
str,str,u32
"""40""","""30""",67366
"""30""","""40""",66090
"""40""","""15""",62291
"""15""","""40""",61843
"""A""","""40""",61295
"""40""","""A""",59656
"""40""","""0""",39804
"""0""","""40""",38325
"""40""","""40""",516


Most standard games end with scores consistent with the provider recording the score immediately before the game-winning point.

The endings `15-0`, `0-30`, and `15-30` are unusual because neither player is yet one point away from winning a standard game and should be investigated for incomplete point-by-point data.

The `40-40` ending is investigated separately because it may represent valid no-ad scoring rather than incomplete data.

### 7.13 Check whether 40-40 endings behave like completed no-ad games

In [61]:
# Build one row per standard game and compare each game with the previous game's set score

standard_game_summary = (
    standard_games
    .sort(
        ["snapshot_date", "match_id", "set_id", "game_id", "point_id"]
    )
    .group_by(
        ["snapshot_date", "match_id", "set_id", "game_id"],
        maintain_order=True
    )
    .agg(
        pl.col("home_point").last().alias("final_home_point"),
        pl.col("away_point").last().alias("final_away_point"),
        pl.col("home_score").first().alias("home_score"),
        pl.col("away_score").first().alias("away_score"),
        pl.col("sequence_issue").first().alias("sequence_issue")
    )
    .sort(["snapshot_date", "match_id", "set_id", "game_id"])
    .with_columns(
        pl.col("home_score")
        .shift(1)
        .over(["snapshot_date", "match_id", "set_id"])
        .alias("prev_home_score"),

        pl.col("away_score")
        .shift(1)
        .over(["snapshot_date", "match_id", "set_id"])
        .alias("prev_away_score")
    )
)

forty_all_games = (
    standard_game_summary
    .filter(
        (pl.col("final_home_point") == "40")
        & (pl.col("final_away_point") == "40")
        & pl.col("sequence_issue").is_null()
    )
    .with_columns(
        (pl.col("home_score") - pl.col("prev_home_score"))
        .alias("home_delta"),

        (pl.col("away_score") - pl.col("prev_away_score"))
        .alias("away_delta")
    )
)

forty_all_games.group_by(
    ["home_delta", "away_delta"]
).len().sort("len", descending=True)

home_delta,away_delta,len
i64,i64,u32
1,0,265
0,1,197
null,null,54


In [62]:
# Check why some 40-40 games have no previous set score

forty_all_games.filter(
    pl.col("home_delta").is_null()
).select(
    [
        "snapshot_date",
        "match_id",
        "set_id",
        "game_id",
        "home_score",
        "away_score"
    ]
).group_by(
    ["game_id", "home_score", "away_score"]
).len().sort("len", descending=True)

game_id,home_score,away_score,len
i64,i64,i64,u32
1,0,1,31
1,1,0,23


The 516 standard games ending at `40-40` are consistent with valid no-ad scoring.

For games with a previous game available, exactly one player's set score increases by one. The remaining cases are first games of a set and end with a set score of `1-0` or `0-1`.

Therefore, `40-40` endings are not classified as invalid because their game-score progression is consistent with completed no-ad scoring.

### 7.14 Investigate unusual incomplete-looking standard games

In [63]:
unusual_standard_games = (
    standard_game_summary
    .filter(
        (
            (pl.col("final_home_point") == "15")
            & (pl.col("final_away_point") == "0")
        )
        |
        (
            (pl.col("final_home_point") == "0")
            & (pl.col("final_away_point") == "30")
        )
        |
        (
            (pl.col("final_home_point") == "15")
            & (pl.col("final_away_point") == "30")
        )
    )
)

unusual_standard_games.select(
    [
        "snapshot_date",
        "match_id",
        "set_id",
        "game_id",
        "final_home_point",
        "final_away_point",
        "home_score",
        "away_score",
        "sequence_issue"
    ]
)

snapshot_date,match_id,set_id,game_id,final_home_point,final_away_point,home_score,away_score,sequence_issue
date,i64,i64,i64,str,str,i64,i64,str
2024-02-13,12064882,2,12,"""15""","""0""",7,5,null
2024-02-14,12064882,2,12,"""15""","""0""",7,5,null
2024-03-04,12126012,1,9,"""0""","""30""",3,6,"""partial_set_start"""
2024-03-05,12126012,1,9,"""0""","""30""",3,6,"""partial_set_start"""
2024-03-17,12166104,2,6,"""15""","""30""",0,6,null


The five remaining unusual standard-game endings represent incomplete point-by-point records.

Their game-level scores indicate that the games were completed, but the recorded point sequences stop at `15-0`, `0-30`, or `15-30`, before either player reaches a legal game-winning position.

These records are retained and flagged as incomplete standard-game point-by-point data. Missing points are not reconstructed.

In [64]:
# Flag standard games whose point sequence ends before a legal game-winning position is reached

incomplete_standard_game_flags = (
    unusual_standard_games
    .select(["snapshot_date", "match_id"])
    .unique()
    .with_columns(
        pl.lit("incomplete_standard_game_end")
        .alias("standard_game_issue")
    )
)

incomplete_standard_game_flags

snapshot_date,match_id,standard_game_issue
date,i64,str
2024-02-13,12064882,"""incomplete_standard_game_end"""
2024-03-17,12166104,"""incomplete_standard_game_end"""
2024-02-14,12064882,"""incomplete_standard_game_end"""
2024-03-04,12126012,"""incomplete_standard_game_end"""
2024-03-05,12126012,"""incomplete_standard_game_end"""


In [65]:
rows_before_standard_join = point_all.height

point_all = point_all.join(
    incomplete_standard_game_flags,
    on=["snapshot_date", "match_id"],
    how="left"
)

print("Rows before join:", rows_before_standard_join)
print("Rows after join:", point_all.height)
print(
    "Row count preserved:",
    rows_before_standard_join == point_all.height
)

Rows before join: 2549369
Rows after join: 2549369
Row count preserved: True


### 7.15 Validate game-to-game set-score progression

In [66]:
# Validate game-to-game set-score progression
# Structurally incomplete match snapshots are excluded to avoid false positives.
# Deciding match tie-breaks are also excluded because their home_score/away_score
# fields use a different encoding.

game_keys = [
    "snapshot_date",
    "match_id",
    "set_id",
    "game_id"
]

game_progression = (
    point_all
    .filter(pl.col("sequence_issue").is_null())
    .group_by(game_keys)
    .agg(
        pl.col("home_score").first().alias("home_score"),
        pl.col("away_score").first().alias("away_score")
    )
    .join(
        other_numeric_games.select(game_keys),
        on=game_keys,
        how="anti"
    )
    .sort(game_keys)
    .with_columns(
        pl.col("home_score")
        .shift(1)
        .over(["snapshot_date", "match_id", "set_id"])
        .alias("prev_home_score"),

        pl.col("away_score")
        .shift(1)
        .over(["snapshot_date", "match_id", "set_id"])
        .alias("prev_away_score")
    )
    .with_columns(
        pl.when(pl.col("game_id") == 1)
        .then(
            (
                (pl.col("home_score") == 1)
                & (pl.col("away_score") == 0)
            )
            |
            (
                (pl.col("home_score") == 0)
                & (pl.col("away_score") == 1)
            )
        )
        .otherwise(
            (
                (
                    pl.col("home_score")
                    - pl.col("prev_home_score")
                ) == 1
            )
            &
            (
                (
                    pl.col("away_score")
                    - pl.col("prev_away_score")
                ) == 0
            )
            |
            (
                (
                    pl.col("home_score")
                    - pl.col("prev_home_score")
                ) == 0
            )
            &
            (
                (
                    pl.col("away_score")
                    - pl.col("prev_away_score")
                ) == 1
            )
        )
        .alias("valid_game_progression")
    )
)

invalid_game_progression = game_progression.filter(
    ~pl.col("valid_game_progression")
)

print(
    "Invalid game-score progressions:",
    invalid_game_progression.height
)

invalid_game_progression.head(20)

Invalid game-score progressions: 0


snapshot_date,match_id,set_id,game_id,home_score,away_score,prev_home_score,prev_away_score,valid_game_progression
date,i64,i64,i64,i64,i64,i64,i64,bool


## 8. Clean and validate the final dataset

### 8.1 Check final key uniqueness and exact duplicates

In [67]:
# Check for exact duplicate rows

exact_duplicates = (
    point_all
    .group_by(point_all.columns)
    .len()
    .filter(pl.col("len") > 1)
)

print("Exact duplicate groups:", exact_duplicates.height)


# Check the natural point-level key

duplicate_point_keys = (
    point_all
    .group_by(
        [
            "snapshot_date",
            "match_id",
            "set_id",
            "game_id",
            "point_id"
        ]
    )
    .len()
    .filter(pl.col("len") > 1)
)

print("Duplicate point keys:", duplicate_point_keys.height)

Exact duplicate groups: 0
Duplicate point keys: 0


### 8.2 Final data-quality validation

In [68]:
# Final validation of the cleaned point-by-point dataset
# Null values in issue columns are intentional and mean "no issue"

issue_columns = [
    "sequence_issue",
    "snapshot_issue",
    "tiebreak_issue",
    "standard_game_issue"
]

source_columns = [
    col for col in point_all.columns
    if col not in issue_columns
]

source_null_count = sum(
    point_all.select(source_columns).null_count().row(0)
)

quality_snapshots = (
    point_all
    .select(
        [
            "snapshot_date",
            "match_id",
            *issue_columns
        ]
    )
    .unique()
)

print("Final rows:", point_all.height)
print("Final columns:", point_all.width)
print("Null values in original/source fields:", source_null_count)

for column in issue_columns:
    count = (
        quality_snapshots
        .filter(pl.col(column).is_not_null())
        .height
    )
    print(f"{column}: {count} flagged match snapshots")

Final rows: 2549369
Final columns: 18
Null values in original/source fields: 0
sequence_issue: 15 flagged match snapshots
snapshot_issue: 1 flagged match snapshots
tiebreak_issue: 9 flagged match snapshots
standard_game_issue: 5 flagged match snapshots


## 9. Save and summarize the cleaned data

In [69]:
# Save the cleaned point-by-point dataset

PROCESSED_ROOT = DATA_ROOT / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

point_output_path = (
    PROCESSED_ROOT
    / "point_by_point_clean.parquet"
)

point_all.write_parquet(point_output_path)

print("Saved to:", point_output_path)
print("Final shape:", point_all.shape)

Saved to: d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data\processed\point_by_point_clean.parquet
Final shape: (2549369, 18)


In [70]:
# Verify that the saved parquet can be read successfully

saved_point_data = pl.read_parquet(point_output_path)

print("Saved shape:", saved_point_data.shape)

print(
    "Exact saved-data match:",
    saved_point_data.equals(point_all)
)

Saved shape: (2549369, 18)
Exact saved-data match: True


### Point-by-point preprocessing summary

The point-by-point dataset was successfully combined, validated, and cleaned while preserving all historical daily snapshots.

Key findings:

- 22,272 point-by-point file appearances were processed.
- The combined dataset contains 2,549,369 point-level observations.
- All source files use a consistent 13-column schema.
- No missing values were present in the original source fields.
- 22,255 files were not stored in chronological point order, so the combined dataset was explicitly sorted by snapshot, match, set, game, and point.
- All recorded `point_id` sequences were unique, consecutive, and started at 0.
- 15 match snapshots contained incomplete game sequences:
  - partial set starts
  - internal missing games
- Repeated daily files were preserved as historical snapshots.
- 59 repeated filenames changed across snapshots:
  - 58 showed cumulative growth
  - 1 contained a genuine historical revision
- Numeric scoring was separated into:
  - standard set tie-breaks
  - deciding match tie-breaks
- All 932 deciding match tie-break observations satisfied the expected minimum 10-point, two-point-margin ending rule.
- 9 standard tie-break snapshots contained incomplete point-by-point endings and were flagged rather than reconstructed.
- 516 games ending at `40-40` were consistent with valid no-ad scoring.
- 5 standard-game snapshots contained incomplete point-level endings and were flagged.
- No exact duplicate rows were found.
- No duplicate point-level keys were found.
- No source rows were deleted and no missing tennis points were reconstructed.

Data-quality issues were retained in explicit flag columns so downstream analysis can distinguish complete observations from incomplete or revised historical records.